Annotated Version of Filter Script

The script below is designed to scrape tweets in the "Pro tier" of twitter, going back to the very first tweet in March of 2006. It is designed to ignore retweets, and duplicates, while collecting the date, text, author, geolocation, total number of likes, and impressions.

In [ ]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_v2(keywords, start_date, end_date, max_tweets=1000000):
    """
    Fetch tweets based on keywords and date range using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    start_date (str): Start date in the format YYYY-MM-DDTHH:mm:ssZ.
    end_date (str): End date in the format YYYY-MM-DDTHH:mm:ssZ.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()
    max_results = 500  # Adjusted to fit within the Pro tier constraints
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=max_results,
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

end_date = (datetime.utcnow() - timedelta(seconds=30)).strftime('%Y-%m-%dT%H:%M:%SZ')
start_date = '2006-03-21T00:00:00Z'  # Start date from the beginning of Twitter

max_tweets = 1000000  # Adjusted for Pro tier

tweets_df = fetch_tweets_v2(keywords, start_date, end_date, max_tweets)

print(tweets_df)
tweets_df.to_csv('FinalTweetPull.csv', index=False)


The script below is designed to scrape tweets in the "Pro tier" of twitter, it can be used to pull tweets from a selected calendar year. It is designed to ignore retweets, and duplicates, while collecting the date, text, author, geolocation, total number of likes, and impressions.

In [ ]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=15000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()
    max_results = 500  # Adjusted to fit within the Pro tier constraints

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=max_results,
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2023

max_tweets = 15000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)


The script below is designed to scrape tweets in the "Pro tier" of twitter, it can be used to pull tweets from a selected range of years. It is designed to ignore retweets, and duplicates, while collecting the date, text, author, geolocation, total number of likes, and impressions.

In [ ]:
import tweepy
import pandas as pd
from datetime import datetime

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_years_range(keywords, start_year, end_year, max_tweets=1000000):
    """
    Fetch tweets based on keywords and a range of years using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    start_year (int): The start year for the range.
    end_year (int): The end year for the range.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()
    max_results = 500  # Adjusted to fit within the Pro tier constraints

    start_date = f'{start_year}-01-01T00:00:00Z'
    end_date = f'{end_year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=max_results,
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the range of years for which to fetch tweets
start_year = 2020
end_year = 2023

max_tweets = 1000000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_years_range(keywords, start_year, end_year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{start_year}_{end_year}.csv', index=False)


The script below is designed to scrape tweets in the "Pro tier" of twitter, it can be used to pull tweets one year back from the moment you run the script. It is designed to ignore retweets, and duplicates, while collecting the date, text, author, geolocation, total number of likes, and impressions. [Need to reconcile with other scripts]

In [14]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
# 'wait_on_rate_limit=True' ensures the client waits if rate limits are hit
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_v2(keywords, start_date, end_date, max_tweets=1000000):
    """
    Fetch tweets based on keywords and date range using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    start_date (str): Start date in the format YYYY-MM-DDTHH:mm:ssZ.
    end_date (str): End date in the format YYYY-MM-DDTHH:mm:ssZ.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    # Build the query string
    # 'lang:en' ensures only English tweets are fetched
    # '-is:retweet' excludes retweets from the results
    query = f'({keywords}) lang:en -is:retweet'
    
    # List to store the tweet data
    tweets_list = []
    
    # Set to track seen tweet IDs and avoid duplicates
    seen_ids = set()

    # Use tweepy's Paginator to handle pagination and fetch tweets
    # 'tweet_fields' specifies the fields we want to include in the tweet data
    for tweet in tweepy.Paginator(client.search_recent_tweets, 
                                  query=query, 
                                  start_time=start_date,
                                  end_time=end_date,
                                  max_results=500,  # Maximum number of results per page
                                  tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']).flatten(limit=max_tweets):
        
        # Check if the tweet ID is already in the set of seen IDs to avoid duplicates
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)  # Add the tweet ID to the set
            
            # Extract the geo information if available
            geo = tweet.geo if 'geo' in tweet.data else None
            
            # Extract public metrics if available
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            # Append the tweet data to the list
            tweets_list.append([
                tweet.created_at,  # Tweet creation date
                tweet.author_id,   # Author ID
                tweet.text,        # Tweet text
                geo,               # Geo information
                impressions,       # Number of impressions
                likes              # Number of likes
            ])
        
    # Convert the list of tweets to a DataFrame
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage

# Define the keywords for the search query
# The query includes exact phrases, hashtags, and combinations of keywords with location constraints
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the date range for the tweets
end_date = (datetime.utcnow() - timedelta(seconds=30)).strftime('%Y-%m-%dT%H:%M:%SZ')
start_date = (datetime.utcnow() - timedelta(days=365)).strftime('%Y-%m-%dT%H:%M:%SZ')

# Define the maximum number of tweets to fetch
max_tweets = 1000000

# Fetch the tweets using the defined parameters
tweets_df = fetch_tweets_v2(keywords, start_date, end_date, max_tweets)

# Print the DataFrame to see the fetched tweets
print(tweets_df)

# Save the DataFrame to a CSV file
tweets_df.to_csv('Filtered_output_tweets_final.csv', index=False)

                        Date                 User  \
0  2024-07-23 16:17:00+00:00            155385137   
1  2024-07-23 10:50:34+00:00              1479081   
2  2024-07-22 19:33:43+00:00           2195439758   
3  2024-07-22 18:09:51+00:00  1336380699121610752   
4  2024-07-22 18:00:17+00:00             14298113   
5  2024-07-22 15:59:22+00:00             20491940   
6  2024-07-22 11:34:47+00:00            339864516   
7  2024-07-22 02:12:37+00:00  1768089224107237377   
8  2024-07-22 01:33:47+00:00  1511793234934042627   
9  2024-07-21 17:31:11+00:00            160094046   
10 2024-07-21 17:27:26+00:00            160094046   
11 2024-07-21 16:49:02+00:00  1001928009706758144   
12 2024-07-21 16:48:05+00:00  1001928009706758144   
13 2024-07-21 16:46:08+00:00  1001928009706758144   
14 2024-07-21 14:44:33+00:00            437068479   
15 2024-07-21 00:46:39+00:00  1427421135495327744   
16 2024-07-21 00:30:02+00:00  1427491022091657216   
17 2024-07-20 18:03:26+00:00            150793

Return here was 18/33 relevant tweets

In [2]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2023

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)


                          Date                 User  \
0    2023-12-31 23:12:04+00:00             42543253   
1    2023-12-31 05:27:28+00:00             43987548   
2    2023-12-31 01:11:27+00:00           2799329899   
3    2023-12-30 15:47:05+00:00  1740088979733807104   
4    2023-12-30 08:27:58+00:00           3272338908   
...                        ...                  ...   
2643 2023-01-01 21:43:16+00:00           2865443107   
2644 2023-01-01 08:05:13+00:00           3193572199   
2645 2023-01-01 07:47:13+00:00  1542690107756404737   
2646 2023-01-01 03:53:58+00:00             26647717   
2647 2023-01-01 01:53:28+00:00   944886953559896064   

                                                  Tweet   Geo  Impressions  \
0     @PinkusCottage Our family has three tortoises!...  None           42   
1     The African spurred tortoise (Centrochelys sul...  None          334   
2     @RedRockCynLV BLM kills more tortoises than an...  None           23   
3     Cal City's tortoise g

In [3]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2022

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date                 User  \
0    2022-12-31 20:15:36+00:00  1609184802011336706   
1    2022-12-31 18:00:23+00:00             28644321   
2    2022-12-31 15:56:12+00:00  1211772607176929280   
3    2022-12-31 15:50:12+00:00  1580212859182624769   
4    2022-12-31 04:28:22+00:00   717885111669121024   
...                        ...                  ...   
3047 2022-01-02 02:25:47+00:00           1486714309   
3048 2022-01-01 20:00:22+00:00             28644321   
3049 2022-01-01 19:48:50+00:00             13958282   
3050 2022-01-01 19:10:27+00:00           1371405440   
3051 2022-01-01 01:42:26+00:00           2360682914   

                                                  Tweet  \
0     Why? He should have been jailed\n\nhttps://t.c...   
1     It’s the final home stretch of 2022 and Mohave...   
2     2022 #HIGHLIGHTS Thrilled to be #UK #Ambassado...   
3     It's aldabra life, Eating lights.  😁✌️😁🥰🥰❣️❤️\...   
4     @HeckinGoodDogs We had so many of thes

In [4]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2021

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date                 User  \
0    2021-12-30 22:35:02+00:00  1394549774968254467   
1    2021-12-30 22:18:35+00:00           1093505790   
2    2021-12-30 21:55:05+00:00  1071495338073059328   
3    2021-12-30 21:06:22+00:00             49496512   
4    2021-12-30 19:32:03+00:00             52129233   
...                        ...                  ...   
2933 2021-01-02 09:05:02+00:00             18953989   
2934 2021-01-02 06:53:20+00:00           2772829754   
2935 2021-01-02 05:42:00+00:00  1229538674330570754   
2936 2021-01-01 19:33:38+00:00             19595223   
2937 2021-01-01 02:42:26+00:00             13958282   

                                                  Tweet  \
0     Turtles are interesting pets to have, but they...   
1     @Dlstoke @DashHound7 @VisitTucsonAZ The desert...   
2     @SecDebHaaland Agreed Secretary Haaland. But n...   
3     Still thinking back to how I visited @_iambam_...   
4     If you wanna learn about the natural w

In [5]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2020

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date                 User  \
0    2020-12-31 20:31:53+00:00             16105779   
1    2020-12-31 20:08:07+00:00             41719383   
2    2020-12-31 14:46:38+00:00  1109178754457165824   
3    2020-12-31 05:18:16+00:00           4853331239   
4    2020-12-31 03:42:10+00:00  1289577604673142784   
...                        ...                  ...   
2982 2020-01-01 16:54:56+00:00  1001928009706758144   
2983 2020-01-01 16:30:26+00:00             64848802   
2984 2020-01-01 07:47:20+00:00  1115783129740926977   
2985 2020-01-01 00:01:19+00:00           2195439758   
2986 2020-01-01 00:01:19+00:00           2195439758   

                                                  Tweet   Geo  Impressions  \
0     @TheBlueGem3 @LouGarza86 A good friend’s daugh...  None            0   
1     "You're in a desert, walking along in the sand...  None            0   
2     Check this out! Of all possible subjects, look...  None            0   
3     Feral Burros and Othe

In [6]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2020

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date                 User  \
0    2020-12-31 20:31:53+00:00             16105779   
1    2020-12-31 20:08:07+00:00             41719383   
2    2020-12-31 14:46:38+00:00  1109178754457165824   
3    2020-12-31 05:18:16+00:00           4853331239   
4    2020-12-31 03:42:10+00:00  1289577604673142784   
...                        ...                  ...   
2982 2020-01-01 16:54:56+00:00  1001928009706758144   
2983 2020-01-01 16:30:26+00:00             64848802   
2984 2020-01-01 07:47:20+00:00  1115783129740926977   
2985 2020-01-01 00:01:19+00:00           2195439758   
2986 2020-01-01 00:01:19+00:00           2195439758   

                                                  Tweet   Geo  Impressions  \
0     @TheBlueGem3 @LouGarza86 A good friend’s daugh...  None            0   
1     "You're in a desert, walking along in the sand...  None            0   
2     Check this out! Of all possible subjects, look...  None            0   
3     Feral Burros and Othe

In [7]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2019

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date                 User  \
0    2019-12-31 21:51:15+00:00            266267158   
1    2019-12-31 21:41:19+00:00            320356232   
2    2019-12-31 19:03:35+00:00           3272338908   
3    2019-12-31 18:51:49+00:00  1198338387343622145   
4    2019-12-31 18:41:04+00:00           2382676303   
...                        ...                  ...   
3339 2019-01-01 11:18:27+00:00             71676966   
3340 2019-01-01 06:03:22+00:00           1385855684   
3341 2019-01-01 05:45:12+00:00           3050083241   
3342 2019-01-01 05:17:30+00:00             82863105   
3343 2019-01-01 02:33:18+00:00            132006871   

                                                  Tweet   Geo  Impressions  \
0     Daisy https://t.co/m1DlpkYjyF #hibernation hop...  None            0   
1     @AiG Oh? Was it awfully snowy in the garden of...  None            0   
2     Gemini #Solar Project final Environmental Impa...  None            0   
3     Anastasia The Desert 

In [8]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2018

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date                User  \
0    2018-12-31 23:07:45+00:00             5845492   
1    2018-12-31 19:47:13+00:00            54096218   
2    2018-12-31 17:32:50+00:00          3284695747   
3    2018-12-31 04:53:45+00:00          2594056208   
4    2018-12-30 21:10:33+00:00            22798511   
...                        ...                 ...   
3571 2018-01-01 16:51:17+00:00           106277663   
3572 2018-01-01 07:00:06+00:00  918963003801907200   
3573 2018-01-01 07:00:06+00:00  918963003801907200   
3574 2018-01-01 06:55:06+00:00  918963003801907200   
3575 2018-01-01 06:55:05+00:00  918963003801907200   

                                                  Tweet  \
0     @cd_hooks Yeah that's the one, it's great. Jus...   
1     #kc365challenge  \nTortoise - The Catastrophis...   
2     #message “the tortoise won, not the hare” @ Oa...   
3     Addison playing with Elliot, the California Su...   
4     @marvelzombiek A better way to find out if you... 

In [9]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2017

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date                User  \
0    2017-12-31 21:58:39+00:00          4909874579   
1    2017-12-31 11:43:09+00:00  741732184868802560   
2    2017-12-31 02:42:15+00:00  775482673179406336   
3    2017-12-31 01:48:44+00:00  775482673179406336   
4    2017-12-31 01:16:30+00:00           473726371   
...                        ...                 ...   
2906 2017-01-01 21:15:11+00:00            26475196   
2907 2017-01-01 20:05:57+00:00           577411640   
2908 2017-01-01 18:46:46+00:00  788727472506408960   
2909 2017-01-01 08:22:01+00:00           205203918   
2910 2017-01-01 07:55:44+00:00           606907135   

                                                  Tweet   Geo  Impressions  \
0          Interesting article\nhttps://t.co/0iEztjgY3A  None            0   
1     I hunger for desert tortoise as I mull over ma...  None            0   
2     @TheLaymansGrit @standbyme44 @Mac_Styli @Murma...  None            0   
3     @TheLaymansGrit @Mac_Styli @Murma

In [10]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2016

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date                User  \
0    2016-12-31 18:50:21+00:00  764556282304966656   
1    2016-12-31 06:34:15+00:00  748262115140476928   
2    2016-12-31 04:22:44+00:00          3281898972   
3    2016-12-30 21:21:49+00:00  799735674538594304   
4    2016-12-29 12:49:49+00:00             5561552   
...                        ...                 ...   
3295 2016-01-03 18:36:00+00:00            57254697   
3296 2016-01-03 05:59:28+00:00            72203607   
3297 2016-01-03 03:35:43+00:00          3700465395   
3298 2016-01-02 07:56:42+00:00          4187562373   
3299 2016-01-01 21:44:46+00:00           959374268   

                                                  Tweet   Geo  Impressions  \
0     10 Beautiful Moment Of Desert TorToise-2017(HD...  None            0   
1     @8BitShawn @Vyse17 Well, you're not wrong. How...  None            0   
2     @johnhurst600 a desert tortoise :3 he's about ...  None            0   
3     Oh the humanity......the carnage.

In [11]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2015

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date        User  \
0    2015-12-31 21:21:56+00:00  1363621915   
1    2015-12-30 20:46:14+00:00    19424173   
2    2015-12-30 20:46:14+00:00   294449141   
3    2015-12-30 01:49:06+00:00  1072302384   
4    2015-12-29 21:44:35+00:00    50716553   
...                        ...         ...   
2983 2015-01-02 22:30:11+00:00   419999807   
2984 2015-01-02 17:45:08+00:00   348448101   
2985 2015-01-02 14:48:44+00:00  2909095652   
2986 2015-01-02 03:43:36+00:00   470078087   
2987 2015-01-01 07:31:53+00:00  2351949139   

                                                  Tweet   Geo  Impressions  \
0     was at vet all day for my desert tortoise had ...  None            0   
1     #8 in 10 Best California Wildlife Moments 2015...  None            0   
2     #8 in 10 Best California Wildlife Moments 2015...  None            0   
3     "I wanna live in Southern California so I can ...  None            0   
4     You might want to check out the new #deserttor...  

In [12]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2014

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date        User  \
0    2014-12-31 12:06:54+00:00   316085667   
1    2014-12-31 08:43:20+00:00    83825919   
2    2014-12-31 00:34:56+00:00   183886857   
3    2014-12-30 14:54:43+00:00   288283415   
4    2014-12-30 06:48:05+00:00   195205347   
...                        ...         ...   
5018 2014-01-01 04:43:24+00:00  1052283055   
5019 2014-01-01 03:01:10+00:00    77365754   
5020 2014-01-01 02:57:04+00:00   130012293   
5021 2014-01-01 02:26:38+00:00   782058314   
5022 2014-01-01 02:26:37+00:00   782058314   

                                                  Tweet  \
0     I wonder how my desert tortoise will react to ...   
1     Perspectic News: 2014 for Carson City, state: ...   
2     Desert tortoise troubles blamed on ravens, oth...   
3     Ask Harry Reid to put his Searchlight 200 acre...   
4     The Egyptian tortoise is a small, desert-livin...   
...                                                 ...   
5018  #Duke's coach has the enthus

In [13]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2013

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date        User  \
0    2013-12-31 12:38:04+00:00   949707872   
1    2013-12-31 08:36:08+00:00   600546355   
2    2013-12-30 20:03:45+00:00   442554494   
3    2013-12-30 18:08:40+00:00    31518842   
4    2013-12-30 14:52:04+00:00  1900629534   
...                        ...         ...   
3723 2013-01-02 00:10:42+00:00    16426616   
3724 2013-01-01 21:15:08+00:00   196538315   
3725 2013-01-01 19:01:28+00:00        1085   
3726 2013-01-01 12:46:35+00:00   273113540   
3727 2013-01-01 06:46:24+00:00   607644971   

                                                  Tweet   Geo  Impressions  \
0     The desert tortoise can live without having to...  None            0   
1     well no i live in a desert more like a uhm des...  None            0   
2     Joshua Tree Study Highlights Climate Threat to...  None            0   
3     If I was sand boarding on a desert island I wo...  None            0   
4     droughts not good for Desert Tortoises and Jos...  

In [14]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2012

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date       User  \
0    2012-12-31 22:56:18+00:00  240922081   
1    2012-12-31 22:49:42+00:00  844304204   
2    2012-12-31 16:01:23+00:00  111052647   
3    2012-12-31 12:48:22+00:00  273113540   
4    2012-12-31 07:11:28+00:00  160477517   
...                        ...        ...   
2409 2012-01-02 20:15:44+00:00   44666696   
2410 2012-01-01 23:29:40+00:00  158956589   
2411 2012-01-01 17:49:24+00:00  119246826   
2412 2012-01-01 05:47:48+00:00  117861977   
2413 2012-01-01 03:16:13+00:00   40413212   

                                                  Tweet   Geo  Impressions  \
0     My Top 2012 desert experiences: Sunset in Deat...  None            0   
1     Great pictures from The Tortoise Cove, and the...  None            0   
2     @Hay You're in a desert and you come across a ...  None            0   
3     Save the Desert Tortoise and its Habitat - Tak...  None            0   
4     Lol @K_G_Bees_knees said I should find a sexy ...  None        

In [15]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2011

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date       User  \
0    2011-12-31 15:25:42+00:00  212858705   
1    2011-12-31 03:42:17+00:00  196458749   
2    2011-12-31 02:15:34+00:00  337830681   
3    2011-12-31 01:48:14+00:00  133887090   
4    2011-12-31 00:50:47+00:00  212858705   
...                        ...        ...   
2224 2011-01-01 13:03:13+00:00   17305036   
2225 2011-01-01 13:03:10+00:00   17190495   
2226 2011-01-01 11:01:33+00:00   14976838   
2227 2011-01-01 05:41:30+00:00   14976838   
2228 2011-01-01 00:21:30+00:00   14976838   

                                                  Tweet  \
0     Mojave Tortoise Doesn't Need Hunting Limits - ...   
1     Mojave Tortoise Doesn't Need Hunting Limits ht...   
2     Mojave Tortoise Doesn't Need Hunting Limits: B...   
3     Mojave Tortoise Doesn't Need Hunting Limits: B...   
4     Mojave Tortoise Doesn't Need Hunting Limits ht...   
...                                                 ...   
2224  100-pound exotic tortoise found in Arizo

In [16]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2010

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                          Date      User  \
0    2010-12-31 19:01:26+00:00  14976838   
1    2010-12-31 16:54:39+00:00  29591372   
2    2010-12-31 13:41:23+00:00  14976838   
3    2010-12-31 08:21:22+00:00  14976838   
4    2010-12-31 03:01:20+00:00  14976838   
...                        ...       ...   
1782 2010-01-01 20:23:41+00:00  18594598   
1783 2010-01-01 20:22:28+00:00  51307600   
1784 2010-01-01 20:22:18+00:00   3554721   
1785 2010-01-01 19:53:45+00:00  26415786   
1786 2010-01-01 16:40:05+00:00  36799679   

                                                  Tweet   Geo  Impressions  \
0     Save the Desert Tortoise and its Habitat  http...  None            0   
1     A tortoise just exploded in the desert. Severe...  None            0   
2     Save the Desert Tortoise and its Habitat  http...  None            0   
3     Save the Desert Tortoise and its Habitat  http...  None            0   
4     Save the Desert Tortoise and its Habitat  http...  None            0   
...

In [17]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2009

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                         Date      User  \
0   2009-12-31 15:46:25+00:00  39387324   
1   2009-12-30 18:53:45+00:00  20744571   
2   2009-12-30 10:54:51+00:00  51125172   
3   2009-12-30 10:40:03+00:00  20744571   
4   2009-12-29 15:46:25+00:00  20848883   
..                        ...       ...   
498 2009-01-28 20:32:36+00:00  17505090   
499 2009-01-27 11:20:43+00:00   3690131   
500 2009-01-22 17:44:51+00:00  14364378   
501 2009-01-20 17:18:18+00:00  19203532   
502 2009-01-14 18:05:33+00:00  14307349   

                                                 Tweet   Geo  Impressions  \
0    @CarrylEdwards Move cattle off desert tortoise...  None            0   
1    Money intended to protect Desert Tortoise habi...  None            0   
2    RT whenpigsflyyy $$ 4 Desert Tortoise habitat ...  None            0   
3    Money intended 4 Desert Tortoise habitat/prote...  None            0   
4    Turbines are careful fit in desert #tortoise #...  None            0   
..                   

In [18]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2008

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                        Date      User  \
0  2008-12-29 20:05:50+00:00  14126628   
1  2008-12-28 22:34:42+00:00  14979791   
2  2008-12-28 21:44:46+00:00  14979791   
3  2008-12-28 21:24:35+00:00  14979791   
4  2008-12-26 10:46:03+00:00  14480224   
..                       ...       ...   
66 2008-04-15 15:05:04+00:00  10879612   
67 2008-04-06 11:32:39+00:00   9059032   
68 2008-03-01 01:49:08+00:00   7794332   
69 2008-02-06 00:34:44+00:00   2061671   
70 2008-01-11 20:25:18+00:00   7049072   

                                                Tweet   Geo  Impressions  \
0   From our Solar Blog Endangered Desert Tortoise...  None            0   
1   flying tortoise - Playing Guitar Hero World To...  None            0   
2   flying tortoise - Playing Guitar Hero World To...  None            0   
3   flying tortoise - Playing Guitar Hero World To...  None            0   
4   @allisonr  Many of the highways in S. Nevada h...  None            0   
..                                     

In [19]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-01-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2007

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

                       Date     User  \
0 2007-10-22 21:40:03+00:00  8386742   
1 2007-10-11 20:00:49+00:00  8386742   
2 2007-08-28 16:37:09+00:00  3949991   
3 2007-07-20 17:29:14+00:00   781641   
4 2007-05-08 13:40:59+00:00  3215861   

                                               Tweet   Geo  Impressions  Likes  
0  Fought with my desert tortoise to bring the bi...  None            0      0  
1  Interrupting my experience to go wrangle a gia...  None            0      0  
2  Life in the slow lane : a desert tortoise tale...  None            0      0  
3  @tofu - Preserve for Big horn sheep, iguanas, ...  None            0      0  
4  Nevada has a state fossil (Ichthyosaur) and a ...  None            0      0  


In [21]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, year, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    year (int): The year for which to fetch tweets.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    start_date = f'{year}-04-01T00:00:00Z'
    end_date = f'{year}-12-31T23:59:59Z'
    
    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the year for which to fetch tweets
year = 2006

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, year, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_{year}.csv', index=False)

Empty DataFrame
Columns: [Date, User, Tweet, Geo, Impressions, Likes]
Index: []


2024 data pull

In [1]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

# Twitter API credentials (replace with your actual bearer token)
bearer_token = 'AAAAAAAAAAAAAAAAAAAAALwnuAEAAAAAzC1kCbrS%2BKH%2FUBlUlcLHfyawwvw%3DuissAbOBIlKYmQ5rslL8BbOns9bJZA0Qihsa1gHAqIqROMDeFK'

# Authenticate with the Twitter API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

def fetch_tweets_by_year(keywords, start_date, end_date, max_tweets=10000):
    """
    Fetch tweets based on keywords and a specific year using Twitter API v2.
    
    Parameters:
    keywords (str): Keywords to search for, following Twitter's advanced query syntax.
    start_date (str): Start date for the tweet search in ISO 8601 format.
    end_date (str): End date for the tweet search in ISO 8601 format.
    max_tweets (int): Maximum number of tweets to fetch.

    Returns:
    DataFrame: DataFrame containing the fetched tweets with columns for date, user, tweet text, geo, impressions, and likes.
    """
    
    query = f'({keywords}) lang:en -is:retweet'
    tweets_list = []
    seen_ids = set()

    paginator = tweepy.Paginator(
        client.search_all_tweets,
        query=query,
        start_time=start_date,
        end_time=end_date,
        max_results=500,  # Adjusted to fit within the Pro tier constraints
        tweet_fields=['created_at', 'text', 'author_id', 'geo', 'public_metrics', 'id']
    )

    for tweet in paginator.flatten(limit=max_tweets):
        if tweet.id not in seen_ids:
            seen_ids.add(tweet.id)
            geo = tweet.geo if 'geo' in tweet.data else None
            impressions = tweet.public_metrics['impression_count'] if 'public_metrics' in tweet.data else None
            likes = tweet.public_metrics['like_count'] if 'public_metrics' in tweet.data else None

            tweets_list.append([
                tweet.created_at,
                tweet.author_id,
                tweet.text,
                geo,
                impressions,
                likes
            ])
    
    tweets_df = pd.DataFrame(tweets_list, columns=['Date', 'User', 'Tweet', 'Geo', 'Impressions', 'Likes'])
    return tweets_df

# Example usage
keywords = (
    '"Mojave Desert Tortoise" OR "Gopherus agassizii" OR "Gopherusagassizii" OR '
    '#Gopherusagassizii OR "Desert Tortoise" OR "DesertTortoise" OR "MojaveDesertTortoise" OR '
    '#MojaveDesertTortoise OR #DesertTortoise OR '
    '(tortoise (mojave OR desert OR California OR Nevada OR Utah OR Arizona))'
)

# Define the start and end dates for fetching tweets in 2024
start_date = f'2024-01-01T00:00:00Z'
end_date = (datetime.utcnow() - timedelta(days=datetime.utcnow().weekday() + 2)).strftime('%Y-%m-%dT23:59:59Z')

max_tweets = 10000  # Adjusted for Pro tier

tweets_df = fetch_tweets_by_year(keywords, start_date, end_date, max_tweets)

print(tweets_df)
tweets_df.to_csv(f'Filtered_output_tweets_2024.csv', index=False)


                          Date                 User  \
0    2024-08-10 23:56:58+00:00   903688921430810624   
1    2024-08-10 23:44:33+00:00            146229474   
2    2024-08-10 22:34:50+00:00           2165203542   
3    2024-08-10 22:31:09+00:00   716054120814092288   
4    2024-08-10 20:09:00+00:00             15309072   
...                        ...                  ...   
1705 2024-01-02 15:07:28+00:00  1597032115161288705   
1706 2024-01-02 13:51:22+00:00           2195439758   
1707 2024-01-02 10:59:54+00:00  1659642287066849285   
1708 2024-01-01 12:23:47+00:00  1613130860223672321   
1709 2024-01-01 02:24:03+00:00             33533303   

                                                  Tweet   Geo  Impressions  \
0     1 Peach Red Cherryhead Redfoot Tortoise Baby (...  None            6   
1     @IrisDCasanova @yashar You're in a desert, wal...  None          122   
2     Large desert tortoise rescued by state trooper...  None           15   
3     @mmpadellan Couldn’t 